# Build the expanded-country panel

Same construction as `build_main_panel` but for DE, IT, FR and ES. Writes `Data/monetary_policy_induced_position_country.csv`, consumed by `analysis/appendix_robustness.do` (Appendix Table A1).

In [ ]:
import pyodbc
import pandas as pd
import numpy as np

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()

# Prep

# Monetary policy shocks

In [ ]:
df = pd.read_csv('C:\\Users\\hermesf\\Projects\\JobMarket\\Data\\bond_timeseries_v2.csv')

In [ ]:
df['Dates'] = pd.to_datetime(df['Dates'])
df['bond_maturity'] = pd.to_datetime(df['bond_maturity'])

In [ ]:
# calculate residual maturity in years
df['residual_bond_maturity'] = ((df['bond_maturity'] - df['Dates']).dt.days / 365)

In [ ]:
df = df[df['residual_bond_maturity'] >= 0]

In [ ]:
df['amt_issued'] = (
    df['amt_issued']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.replace(' ', '', regex=False)
)

# convert to float, coercing invalid entries to NaN
df['amt_issued'] = pd.to_numeric(df['amt_issued'], errors='coerce')
df['amt_issued'] = df['amt_issued']/1e9

In [ ]:
df['amt_issued'].fillna(df['amt_issued'].median(), inplace=True)

In [ ]:
df['collateral_country'] = df['ISIN'].str[:2]

In [ ]:
df = df[df['collateral_country'].isin(['DE','IT', 'ES', 'FR'])]

In [ ]:
securities = tuple(df['ISIN'].unique())

In [ ]:
# Data prep
query = f""" 

SELECT s.business_date, 
security_isin as isin,
sum(nominal_euro) as borrowing_volume,
avg(contractual_maturity) as long_maturity,
avg(haircut) as long_haircut

FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s 
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01' 
AND nominal_ccy IN ('EUR') 
AND central_clearing = 'non-cleared' 
AND borrower_country_residence = 'KY' AND s_borrower.sector = 'IF'
AND gnlcoll = 'SPEC'
AND security_isin IN {securities}
--AND contractual_maturity <= 1
GROUP BY business_date, isin
ORDER BY business_date, isin 

""" 

df_borrowing = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f""" 
 

SELECT s.business_date,
security_isin as isin,
sum(nominal_euro) as lending_volume,
avg(contractual_maturity) as short_maturity,
avg(haircut) as short_haircut

FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s 
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01'  
AND nominal_ccy IN ('EUR') 
AND central_clearing = 'non-cleared' 
AND lender_country_residence = 'KY' AND s_lender.sector = 'IF'
AND gnlcoll = 'SPEC'
AND security_isin IN {securities}
--AND contractual_maturity <= 1
GROUP BY business_date, isin
ORDER BY business_date, isin 

""" 

df_lending = pd.read_sql_query(query, cnxn)

In [ ]:
df_repo = df_borrowing.merge( df_lending, on=['business_date', 'isin'], how='outer', suffixes=('_borrowing', '_lending') )
 

In [ ]:
df_repo['business_date'] = pd.to_datetime(df_repo['business_date'])

In [ ]:
df = (
    df[['Dates', 'ISIN', 'PX_ASK', 'PX_BID', 'YLD_YTM_ASK', 'YLD_YTM_BID', 'amt_issued', 'residual_bond_maturity', 'collateral_country']]
    .merge(df_repo, right_on=['business_date','isin'], left_on = ['Dates', 'ISIN'], how='left')
)

In [ ]:
df = df[df['PX_ASK'].isna() == False]

In [ ]:
df.drop(columns=['business_date', 'isin'], inplace=True)

In [ ]:
df = df.rename(columns={'Dates': 'business_date'})
df = df.rename(columns={'ISIN': 'isin'})

In [ ]:
df['borrowing_volume'].fillna(0, inplace = True)
df['lending_volume'].fillna(0, inplace = True)

In [ ]:
df['net_pos'] = (df['borrowing_volume'] - df['lending_volume'])/1e9

In [ ]:
shocks_all = pd.read_excel('C:\\Users\\hermesf\\Projects\\JobMarket\\Data\\altavilla.xlsx', sheet_name = 'Monetary Event Window')

In [ ]:
shocks_all['date'] = pd.to_datetime(shocks_all['date'], dayfirst=True)

In [ ]:
out = (
    df.merge(shocks_all[['date', 'OIS_2Y']], left_on=['business_date'], right_on=['date'], how='left')
       .drop(columns=['date'])  # helpers
)


In [ ]:
out['OIS_2Y'].fillna(0, inplace=True)

In [ ]:
out = out.sort_values(['isin','business_date'])

# mid yield
out['yld_mid'] = (out['YLD_YTM_BID'] + out['YLD_YTM_ASK']) / 2

# ISIN-level daily change
out['delta_y'] = out.groupby('isin')['yld_mid'].diff()

In [ ]:
# 1. Base Intensity: Rolling mean of |net_pos| (Magnitude only)
out['abs_net'] = (out['net_pos'].abs() / out['amt_issued']) * 100

In [ ]:
hf_roll = (
    out.groupby('isin')['abs_net']
     .apply(lambda s: s.rolling(window=5, min_periods=3).mean().shift(1))
     .reset_index(level=0, drop=True)   # <-- align index with d
)

out['hf_intensity_pre'] = hf_roll

In [ ]:
# Calculate the generic intensity (magnitude) first
hf_roll_magnitude = (
    out.groupby('isin')['abs_net']
      .apply(lambda s: s.rolling(window=5, min_periods=3).mean().shift(1))
      .reset_index(level=0, drop=True)
)

# 2. Determine Direction at t-1
# We need to know if they were Long or Short entering the day
out['prev_net_pos'] = out.groupby('isin')['net_pos'].shift(1)

# 3. Create the Two Split Variables
# Variable A: Fragile (Net Long)
out['hf_intensity_long'] = np.where(
    out['prev_net_pos'] > 0,   # Condition: Were they Net Long yesterday?
    hf_roll_magnitude,               # If Yes: Keep the intensity
    0                                # If No: Set to 0
)

# Variable B: Robust (Net Short)
out['hf_intensity_short'] = np.where(
    out['prev_net_pos'] < 0,   # Condition: Were they Net Short yesterday?
    hf_roll_magnitude,               # If Yes: Keep the intensity
    0                                # If No: Set to 0
)

In [ ]:
out['bid_ask_spread'] = out['PX_ASK'] - out['PX_BID'] 

In [ ]:
out['delta_y'] = out['delta_y']*100

In [ ]:
ctd = pd.read_excel('C:\\Users\\hermesf\\Projects\\JobMarket\\Data\\CTD.xlsx')

In [ ]:
# month-end of (Year, Month)
ctd['period_end'] = pd.to_datetime(dict(year=ctd['Year'], month=ctd['Month'], day=1)) + pd.offsets.MonthEnd(0)
ctd['period_start'] = ctd['period_end'] - pd.DateOffset(years=1)

In [ ]:
# --- expand & mark in-window ---
m = out.merge(ctd[['isin','period_start','period_end']], on='isin', how='left')
m['in_window'] = (m['business_date'] >= m['period_start']) & (m['business_date'] <= m['period_end'])

# collapse back to one row per (ISIN, business_date): 1 if any window matches
flag_df = (m.groupby(['isin','business_date'], as_index=False)['in_window']
             .any()
             .rename(columns={'in_window':'ctd_flag'}))

# --- attach flag to original `out` ---
out = out.merge(flag_df, on=['isin','business_date'], how='left')
out['ctd_flag'] = out['ctd_flag'].fillna(False).astype(int)


In [ ]:
# create binary
out.loc[(out['hf_intensity_pre'] > 0), 'hf_involved'] = 1
out['hf_involved'].fillna(0, inplace = True)

In [ ]:
# create binary
out.loc[(out['prev_net_pos'] > 0), 'hf_involved_long'] = 1
out['hf_involved_long'].fillna(0, inplace = True)

out.loc[(out['prev_net_pos'] < 0), 'hf_involved_short'] = 1
out['hf_involved_short'].fillna(0, inplace = True)

In [ ]:
out['hf_intensity_pre'][out['hf_intensity_pre'] > 0].quantile(0.99)

In [ ]:
out['hf_intensity_pre'][out['hf_intensity_pre'] > 0].quantile(0.01)

In [ ]:
out = out[out['hf_intensity_pre'] < 18]

In [ ]:
# Four categories: None / Low / Medium / High
out['hf_category'] = 'ANone'

# Identify ISIN–days with positive HF activity
mask = out['hf_intensity_pre'].fillna(0) > 0

# Within each country, split only those into terciles
out.loc[mask, 'hf_category'] = (
    out.loc[mask]
       .groupby('collateral_country')['hf_intensity_pre']
       .transform(lambda x: pd.qcut(x, 2, labels=['Low', 'High']))
       .astype(str)
)

In [ ]:
# Four categories: None / Low / Medium / High
out['hf_category_ext'] = 'ANone'

# Identify ISIN–days with positive HF activity
mask = out['hf_intensity_pre'].fillna(0) > 0

# Within each country, split only those into terciles
out.loc[mask, 'hf_category_ext'] = (
    out.loc[mask]
       .groupby('collateral_country')['hf_intensity_pre']
       .transform(lambda x: pd.qcut(x, 3, labels=['Low', 'Medium', 'High']))
       .astype(str)
)

In [ ]:
out = out[out['delta_y'].isna() == False]

In [ ]:
out['delta_y'].quantile(0.999)

In [ ]:
out['delta_y'].quantile(0.001)

In [ ]:
out = out[(out['delta_y'] <= 40) & (out['delta_y'] >= -40)]

In [ ]:
germany = pd.read_csv(r'C:\Users\hermesf\Projects\JobMarket\Data\NelsonSiegel_DE_prices.csv')

In [ ]:
italy = pd.read_csv(r'C:\Users\hermesf\Projects\JobMarket\Data\NelsonSiegel_IT_prices.csv')

In [ ]:
spain = pd.read_csv(r'C:\Users\hermesf\Projects\JobMarket\Data\NelsonSiegel_ES_prices.csv')

In [ ]:
france = pd.read_csv(r'C:\Users\hermesf\Projects\JobMarket\Data\NelsonSiegel_FR_prices.csv')

In [ ]:
df_duration = pd.concat([germany, italy, spain, france], ignore_index=True)

In [ ]:
df_duration['refdate'] = pd.to_datetime(df_duration['refdate'])

In [ ]:
out = out.merge(df_duration[['refdate', 'bondcode', 'duration']], left_on = ['business_date', 'isin'], right_on = ['refdate', 'bondcode'], how = 'left')

In [ ]:
out['duration'].fillna(out['residual_bond_maturity'], inplace = True)

In [ ]:
summary = out.groupby('hf_category').agg({
    'isin': 'count',
    'collateral_country': lambda x: (x == 'IT').mean(),
    'amt_issued': 'mean',
    'residual_bond_maturity': 'mean',
    'bid_ask_spread': 'mean',
    'ctd_flag': 'mean'
})

print(summary)

In [ ]:
jk_shock = pd.read_csv('C:\\Users\\hermesf\\Projects\\JobMarket\\Data\\shocks_ecb_mpd_me_d.csv')

In [ ]:
jk_shock['business_date'] = pd.to_datetime(jk_shock['date'])

In [ ]:
out = out.merge(jk_shock, on = 'business_date', how = 'left')

In [ ]:
# Option 1: reassign
out[['pc1', 'STOXX50', 'MP_pm', 'CBI_pm', 'MP_median', 'CBI_median']] = out[['pc1', 'STOXX50', 'MP_pm', 'CBI_pm', 'MP_median', 'CBI_median']].fillna(0)

In [ ]:
out[['pc1', 'STOXX50', 'MP_pm', 'CBI_pm', 'MP_median', 'CBI_median']] = out[['pc1', 'STOXX50', 'MP_pm', 'CBI_pm', 'MP_median', 'CBI_median']] * 100

In [ ]:
# 1. Calculate the daily change in position (Flow)
out['daily_net_change'] = out.groupby('isin')['net_pos'].diff()
out['delta_intensity'] = (out['daily_net_change'] / out['amt_issued']) * 100

In [ ]:
# 3. Determine the Direction ENTERING the shock (State) - Keep this as is
out['is_long_pre'] = (out.groupby('isin')['net_pos'].shift(1) > 0).astype(int)
out['is_short_pre'] = (out.groupby('isin')['net_pos'].shift(1) < 0).astype(int)

In [ ]:
out['placebo_shock'] = out.groupby('isin')['OIS_2Y'].shift(15)
out['placebo_shock'].fillna(0, inplace=True)

In [ ]:
out.to_csv(r'C:\Users\hermesf\Projects\JobMarket\Data\monetary_policy_induced_position_country.csv')